<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/GPT4o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai

In [6]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [5]:
##load dataset
import pandas as pd

url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")

print(df.shape)
df.head()


(12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:

set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].astype(str).str.replace("\n", " ", regex=False).str.strip()

print(set1.shape)
set1.head()

(1783, 3)


,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [8]:
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)

set1["domain1_score"].value_counts().sort_index()

,count
domain1_score,
2,10
3,1
4,17
5,17
6,110
7,135
8,687
9,334
10,316


In [11]:
RANDOM_STATE = 42

calibration_pool = (
    set1.groupby("domain1_score", group_keys=False)
    .apply(lambda x: x.sample(min(1, len(x)), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

calibration = calibration_pool.sample(6, random_state=RANDOM_STATE).reset_index(drop=True)

calibration

/tmp/ipykernel_8115/73661984.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(1, len(x)), random_state=RANDOM_STATE))


,essay_id,essay,domain1_score,score_category
0,1277,"Dear @CAPS1 @CAPS2, I am writing you this lett...",7,Medium
1,1592,The computers are cool. Do you now I werpsite ...,2,Low
2,1304,Would you like to discover new things and expl...,11,High
3,953,"Dear @CAPS1 of @LOCATION1, @CAPS2 the world ha...",12,High
4,19,I aegre waf the evansmant ov tnachnolage. The ...,4,Low
5,22,Dear local Newspaper @CAPS1 a take all your co...,3,Low


In [12]:
## removing calibration from dataset
calibration_ids = calibration["essay_id"].tolist()
pool = set1[~set1["essay_id"].isin(calibration_ids)].copy().reset_index(drop=True)

In [13]:
##Sample TEST SET (30 essays)
demo_30 = pool.sample(30, random_state=RANDOM_STATE).reset_index(drop=True)

print("Calibration:", calibration.shape)
print("Demo 30:", demo_30.shape)

Calibration: (6, 4)
Demo 30: (30, 4)


In [14]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [16]:
##calibration format with ids
def format_calibration_examples_with_id(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [17]:
##building the prompt
def build_mts_prompt_enhanced(essay_id, essay_text, rubric_text, calibration_text_with_id):
    return f"""

{rubric_text}
Note: Final domain1_score = rater1(1-6) + rater2(1-6), so final scores range from 2 to 12.

Content = relevance of ideas, argument development, supporting details
Organization = structure, coherence, logical flow, paragraphing
Language = grammar, vocabulary, clarity, sentence control

{calibration_text_with_id}

Pattern:
- Low (2-5) = weak development and weak organization
- Medium (6-8) = adequate support and reasonable structure
- High (9-12) = strong reasoning, development, and structure

Essay ID: {essay_id}

Essay:
{essay_text}

1. Find the closest calibration example by overall writing quality.
2. Judge Content, Organization, and Language separately using the rubric.
3. Use the full trait range from 1 to 6 when justified.
4. Strong essays should receive 5s or 6s on multiple traits.
5. Return valid JSON only.

{{
  "essay_id": {essay_id},
  "content": X,
  "organization": X,
  "language": X,
  "reasoning": "2 short sentences"
}}

"""

In [18]:
##JSON parser
import json
import re

def parse_json_output(text):
    if not isinstance(text, str):
        return {}

    try:
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    return {}

In [ ]:
##Compute final score
import numpy as np

def compute_mts_score(content, organization, language):
    if None in [content, organization, language]:
        return None, None

    avg = np.mean([content, organization, language])
    final_score = int(round(avg * 2))
    final_score = max(2, min(12, final_score))

    if final_score <= 5:
        category = "Low"
    elif final_score <= 8:
        category = "Medium"
    else:
        category = "High"

    return final_score, category